In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install --upgrade --force-reinstall numpy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 114.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 126.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 59.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfully uninstalled threadpoolctl-3.6.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
  Attemptin

In [3]:
!pip install numpy==1.26.4 -q
!pip install matplotlib==3.8.4 -q
!pip install ultralytics==8.3.35 -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requ

In [4]:
# Exécuter cette cellule EN PREMIER
!pip uninstall -y ray ray[default] ray[tune] -q
print("✅ Ray désinstallé")

✅ Ray désinstallé


#  Imports et Configuration

In [5]:
import os
import pandas as pd
import shutil
import ast
import yaml
import random
import numpy as np
import torch
import gc
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
# Chemins
train_images_path = "/kaggle/input/global-wheat-detection/train"
train_csv_path = "/kaggle/input/global-wheat-detection/train.csv"
base_output_path = "/kaggle/working/GWHD_YOLO_LODO"
os.makedirs(base_output_path, exist_ok=True)


In [9]:
# Hyperparamètres
IMG_SIZE = 512
BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 10
SEED = 42

In [10]:
# Seeds pour reproductibilité
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Fichier résultats
metrics_csv = os.path.join(base_output_path, "results.csv")
with open(metrics_csv, "w") as f:
    f.write("fold,test_country,train_domains,n_train,n_val,n_test,map50,map50_95\n")

In [13]:
# Fonctions utilitaires
# ==============================
def convert_bbox_to_yolo(bbox, img_width, img_height):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return [0, x_center, y_center, w_norm, h_norm]

def create_labels_and_images(df_subset, output_dir, split_name):
    img_dir = os.path.join(output_dir, "images", split_name)
    label_dir = os.path.join(output_dir, "labels", split_name)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    for img_id, group in df_subset.groupby('image_id'):
        img_filename = img_id + ".jpg"
        src = os.path.join(train_images_path, img_filename)
        dst = os.path.join(img_dir, img_filename)

        if os.path.exists(src) and not os.path.exists(dst):
            try:
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)

        label_path = os.path.join(label_dir, img_id + ".txt")
        with open(label_path, "w") as f:
            for _, row in group.iterrows():
                bbox = ast.literal_eval(row['bbox'])
                yolo_bbox = convert_bbox_to_yolo(bbox, row['width'], row['height'])
                f.write(" ".join(map(lambda x: f"{x:.6f}", yolo_bbox)) + "\n")
    return df_subset['image_id'].nunique()

In [14]:
# ==============================
# Lecture du CSV
# ==============================
df = pd.read_csv(train_csv_path)
countries = df['source'].unique()
print("Pays disponibles :", countries)


Pays disponibles : ['usask_1' 'arvalis_1' 'inrae_1' 'ethz_1' 'arvalis_3' 'rres_1' 'arvalis_2']


In [16]:
# ==============================
# Préparer les folds LODO (une seule fois)
# ==============================
fold_info = []

for fold_idx, test_country in enumerate(countries, 1):
    df_test = df[df['source'] == test_country]
    df_train_full = df[df['source'] != test_country]

    df_train_list, df_val_list = [], []
    for country in df_train_full['source'].unique():
        df_c = df_train_full[df_train_full['source'] == country]
        img_ids = df_c['image_id'].unique().tolist()
        random.shuffle(img_ids)
        n_val = int(0.1 * len(img_ids))
        val_ids = img_ids[:n_val]
        df_val_list.append(df_c[df_c['image_id'].isin(val_ids)])
        df_train_list.append(df_c[~df_c['image_id'].isin(val_ids)])

    df_train = pd.concat(df_train_list)
    df_val = pd.concat(df_val_list)

    fold_dir = os.path.join(base_output_path, f"fold_{fold_idx}_{test_country}")
    os.makedirs(fold_dir, exist_ok=True)

    n_train = create_labels_and_images(df_train, fold_dir, "train")
    n_val = create_labels_and_images(df_val, fold_dir, "val")
    n_test = create_labels_and_images(df_test, fold_dir, "test")

    yaml_path = os.path.join(base_output_path, f"fold_{fold_idx}_{test_country}.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump({
            "path": fold_dir,
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": 1,
            "names": ["wheat"]
        }, f)

    fold_info.append({
        'fold_idx': fold_idx,
        'test_country': test_country,
        'train_domains': [c for c in countries if c != test_country],
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'yaml_path': yaml_path
    })

print("✅ Tous les folds préparés !")


✅ Tous les folds préparés !


# Cellule 1 – Fold 1

In [17]:
fold_idx_to_train = 1
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 1
   Test Country : usask_1
   Train Countries : arvalis_1, inrae_1, ethz_1, arvalis_3, rres_1, arvalis_2



100%|██████████| 6.25M/6.25M [00:00<00:00, 108MB/s]


New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_1_usask_1.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_1, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=False, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=F

100%|██████████| 755k/755k [00:00<00:00, 28.0MB/s]
E0000 00:00:1766141431.686553      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766141431.738329      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766141432.156570      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766141432.156610      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766141432.156613      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766141432.156615      55 computation_placer.cc:177] comput

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_1_usask_1/labels/test... 200 images, 0 backgrounds, 0 corrupt: 100%|██████████| 200/200 [00:01<00:00, 161.84it/s]


val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_1_usask_1/labels/test.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  4.38it/s]

                   all        200       5807      0.874       0.84       0.89      0.387
Speed: 0.1ms preprocess, 1.9ms inference, 0.0ms loss, 0.9ms postprocess per image



✅ Résultats Fold 1 : mAP50=0.8901, mAP50-95=0.3871


# fin fold 1


# Fold 2

In [18]:
fold_idx_to_train = 2
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 2
   Test Country : arvalis_1
   Train Countries : usask_1, inrae_1, ethz_1, arvalis_3, rres_1, arvalis_2

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_2, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/train... 2296 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2296/2296 [00:01<00:00, 1324.56it/s]


train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/train.cache


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/val... 436 images, 0 backgrounds, 0 corrupt: 100%|██████████| 436/436 [00:00<00:00, 1396.34it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


...

val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/test... 1055 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1055/1055 [00:00<00:00, 1343.36it/s]


val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_2_arvalis_1/labels/test.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 66/66 [00:14<00:00,  4.42it/s]

                   all       1055      45716      0.841      0.777      0.841      0.338
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 1.0ms postprocess per image



✅ Résultats Fold 2 : mAP50=0.8408, mAP50-95=0.3384


# fin fold 2


# fold3

In [19]:
fold_idx_to_train = 3
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 3
   Test Country : inrae_1
   Train Countries : usask_1, arvalis_1, ethz_1, arvalis_3, rres_1, arvalis_2

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_3, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=3

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/train... 3167 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3167/3167 [00:02<00:00, 1371.31it/s]


train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/train.cache


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/val... 604 images, 0 backgrounds, 0 corrupt: 100%|██████████| 604/604 [00:00<00:00, 1384.93it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


...

Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/test... 176 images, 0 backgrounds, 0 corrupt: 100%|██████████| 176/176 [00:00<00:00, 1388.12it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_3_inrae_1/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:02<00:00,  5.25it/s]


                   all        176       3701      0.944      0.897      0.961      0.586
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 0.8ms postprocess per image

✅ Résultats Fold 3 : mAP50=0.9610, mAP50-95=0.5861


# fin fold 3


# Fold4

In [20]:
fold_idx_to_train = 4
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 4
   Test Country : ethz_1
   Train Countries : usask_1, arvalis_1, inrae_1, arvalis_3, rres_1, arvalis_2

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_4, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=30

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/train... 2597 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2597/2597 [00:02<00:00, 1271.59it/s]

train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/train.cache



val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/val... 491 images, 0 backgrounds, 0 corrupt: 100%|██████████| 491/491 [00:00<00:00, 1387.54it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/val.cache


...

                   all        491      17902      0.919      0.889      0.946       0.54
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/test... 747 images, 0 backgrounds, 0 corrupt: 100%|██████████| 747/747 [00:00<00:00, 1174.45it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_4_ethz_1/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 47/47 [00:09<00:00,  4.73it/s]


                   all        747      51489      0.886      0.838      0.875      0.363
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 1.1ms postprocess per image

✅ Résultats Fold 4 : mAP50=0.8749, mAP50-95=0.3632


# Fold5

In [21]:
fold_idx_to_train = 5
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 5
   Test Country : arvalis_3
   Train Countries : usask_1, arvalis_1, inrae_1, ethz_1, rres_1, arvalis_2

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_5, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/train... 2790 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2790/2790 [00:02<00:00, 1329.25it/s]


train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/train.cache


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/val... 534 images, 0 backgrounds, 0 corrupt: 100%|██████████| 534/534 [00:00<00:00, 1370.89it/s]


val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 512 train, 512 val
Using 0 dataloader workers
Logging results to /kaggle/working/GWHD_YOLO_LODO/YOLOv8_results/fold_5
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/test... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<00:00, 1112.30it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_5_arvalis_3/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:07<00:00,  4.93it/s]


                   all        559      16665      0.897      0.853      0.919      0.473
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 0.9ms postprocess per image

✅ Résultats Fold 5 : mAP50=0.9187, mAP50-95=0.4733


# Fold6

In [22]:
fold_idx_to_train = 6
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 6
   Test Country : rres_1
   Train Countries : usask_1, arvalis_1, inrae_1, ethz_1, arvalis_3, arvalis_2

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_6, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=30

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/train... 2913 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2913/2913 [00:02<00:00, 1345.64it/s]


train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/train.cache


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/val... 554 images, 0 backgrounds, 0 corrupt: 100%|██████████| 554/554 [00:00<00:00, 1274.96it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


...

                   all        554      24162      0.917      0.886      0.941       0.55
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/test... 432 images, 0 backgrounds, 0 corrupt: 100%|██████████| 432/432 [00:00<00:00, 1328.08it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_6_rres_1/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 27/27 [00:05<00:00,  5.25it/s]


                   all        432      20236      0.937      0.873      0.939      0.474
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 0.9ms postprocess per image

✅ Résultats Fold 6 : mAP50=0.9392, mAP50-95=0.4738


# fin fold 6

# Fold7

In [23]:
fold_idx_to_train = 7
fold = fold_info[fold_idx_to_train - 1]

print(f"\n{'='*60}")
print(f"🚀 DÉBUT ENTRAÎNEMENT FOLD {fold_idx_to_train}")
print(f"   Test Country : {fold['test_country']}")
print(f"   Train Countries : {', '.join(fold['train_domains'])}")
print(f"{'='*60}\n")

import gc, torch
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=fold['yaml_path'],
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    seed=SEED,
    project=os.path.join(base_output_path, "YOLOv8_results"),
    name=f"fold_{fold_idx_to_train}",
    cache=False,
    workers=0,
    plots=False,
    save_period=-1,
    verbose=True,
    pretrained=True,
    exist_ok=True,
    device=0,
    augment=False,
    optimizer='auto',
    amp=True
)

metrics = model.val(split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False, plots=False, save_json=False)
print(f"\n✅ Résultats Fold {fold_idx_to_train} : mAP50={metrics.box.map50:.4f}, mAP50-95={metrics.box.map:.4f}")

del model, results
gc.collect()
torch.cuda.empty_cache()



🚀 DÉBUT ENTRAÎNEMENT FOLD 7
   Test Country : arvalis_2
   Train Countries : usask_1, arvalis_1, inrae_1, ethz_1, arvalis_3, rres_1

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=512, save=True, save_period=-1, cache=False, device=0, workers=0, project=/kaggle/working/GWHD_YOLO_LODO/YOLOv8_results, name=fold_7, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det

train: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/train... 3139 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3139/3139 [00:02<00:00, 1351.84it/s]


train: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/train.cache


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/val... 598 images, 0 backgrounds, 0 corrupt: 100%|██████████| 598/598 [00:00<00:00, 1402.58it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


...

Ultralytics 8.3.35 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/test... 204 images, 0 backgrounds, 0 corrupt: 100%|██████████| 204/204 [00:00<00:00, 1335.03it/s]

val: New cache created: /kaggle/working/GWHD_YOLO_LODO/fold_7_arvalis_2/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  5.05it/s]


                   all        204       4179      0.816      0.755      0.826      0.343
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 0.9ms postprocess per image

✅ Résultats Fold 7 : mAP50=0.8256, mAP50-95=0.3433


# fin fold 7

In [29]:
# Extraire métriques
mAP50 = metrics.box.map50
mAP5095 = metrics.box.map

print(f"\n✅ Résultats Fold {fold_idx_to_train} : "
      f"mAP50={mAP50:.4f}, mAP50-95={mAP5095:.4f}")

# ======================
# Sauvegarde CSV
# ======================
with open(metrics_csv, "a") as f:
    train_domains_str = "|".join(fold['train_domains'])
    f.write(
        f"{fold['fold_idx']},{fold['test_country']},{train_domains_str},"
        f"{fold['n_train']},{fold['n_val']},{fold['n_test']},"
        f"{mAP50:.4f},{mAP5095:.4f}\n"
    )


✅ Résultats Fold 7 : mAP50=0.8256, mAP50-95=0.3433


In [27]:
#mAP50 = metrics.box.map50
#mAP5095 = metrics.box.map
